# 🌾 CropMandi AI – 3-Day Farmer Mandi Price Prediction System
## Complete Machine Learning Notebook (Colab Ready)

This notebook implements the complete end-to-end Machine Learning pipeline for **CropMandi AI**:
1. **Data Loading & Validation**: Handles official CSV mandi price reports.
2. **Data Cleaning & Normalization**: Canonicalizes market names, districts, and commodities.
3. **Time-Series Feature Engineering**: Generates lag features, rolling windows, cyclical time encodings, and regional stats.
4. **Chronological Splitting**: Train (2021–2025) and Test (2026) splitting to prevent data leakage.
5. **Multi-Horizon CatBoost Training**: Direct forecasting models for **Horizon 1 (Day 1)**, **Horizon 2 (Day 2)**, and **Horizon 3 (Day 3)**.
6. **Conformal Uncertainty Quantification**: Computes empirical 80% confidence lower & upper bounds.
7. **Model Evaluation & Visualizations**: MAE, RMSE, MAPE, WAPE, sMAPE, R², Interval Coverage, and Feature Importance.
8. **Inference & Decision Advisory**: Generates 3-day forecasts with actionable farmer guidance.

### Step 1: Install & Import Required Packages

In [ ]:
!pip install -q catboost lightgbm pandas numpy scikit-learn matplotlib seaborn plotly

import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styles
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Environment initialized successfully.")

### Step 2: Load Dataset
*Upload your CSV dataset file (e.g. `All_Type_of_Report_(All_Grades)...csv`) or run sample data creation below if testing in a fresh environment.*

In [ ]:
try:
    from google.colab import files
    print("Please upload your CropMandi CSV dataset file:")
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
    print(f"✅ File '{file_path}' uploaded successfully.")
except Exception as e:
    print("Not running in Google Colab environment or no file uploaded. Looking for local file...")
    file_path = "All_Type_of_Report_(All_Grades)_12-08-2026_04-10-30_PM.csv"
    if not os.path.exists(file_path):
        print(f"⚠️ '{file_path}' not found locally. Generating synthetic mandi price sample for demonstration...")
        # Generate synthetic dataset for demonstration
        dates = pd.date_range(start="2023-01-01", end="2026-08-10", freq="D")
        markets = ["Madanapalli APMC", "Kalikiri APMC", "Anantapur APMC", "Punganur APMC"]
        rows = []
        for m in markets:
            base_price = 1500 if "Madanapalli" in m else 1400
            for d in dates:
                price = base_price + np.sin(d.dayofyear / 365.0 * 2 * np.pi) * 400 + np.random.normal(0, 50)
                rows.append({
                    'State': 'Andhra Pradesh',
                    'District': 'Annamayya' if 'Madanapalli' in m or 'Kalikiri' in m else 'Anantapur',
                    'Market': m,
                    'Commodity': 'Tomato',
                    'Arrival_Date': d.strftime('%d/%m/%Y'),
                    'Min Price': max(200, price - 150),
                    'Max Price': price + 150,
                    'Modal Price': price,
                    'Arrivals (Tones)': np.random.uniform(20, 150)
                })
        df_raw = pd.DataFrame(rows)
        df_raw.to_csv("synthetic_mandi_data.csv", index=False)
        file_path = "synthetic_mandi_data.csv"

print(f"Using data file: {file_path}")
df_raw = pd.read_csv(file_path)
print(f"Raw dataset dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()

### Step 3: Data Cleaning & Normalization Engine

In [ ]:
def clean_and_normalize_data(df):
    df_clean = df.copy()
    
    # Standardize column headers
    col_map = {
        'State': 'state',
        'District': 'district',
        'Market': 'market',
        'Commodity': 'commodity',
        'Arrival_Date': 'observation_date',
        'Min Price': 'min_price',
        'Max Price': 'max_price',
        'Modal Price': 'modal_price',
        'Arrivals (Tones)': 'arrival_quantity'
    }
    df_clean.rename(columns=col_map, inplace=True)
    
    # Parse observation dates safely
    df_clean['observation_date'] = pd.to_datetime(df_clean['observation_date'], format='%d/%m/%Y', errors='coerce')
    df_clean.dropna(subset=['observation_date', 'modal_price', 'market', 'commodity'], inplace=True)
    
    # Clean numeric price columns
    for col in ['min_price', 'max_price', 'modal_price', 'arrival_quantity']:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            
    # Filter out unreasonable price outliers (< ₹100 or > ₹15,000 per qtl)
    df_clean = df_clean[(df_clean['modal_price'] >= 100) & (df_clean['modal_price'] <= 15000)]
    
    # Strip whitespace & capitalize canonical names
    df_clean['market'] = df_clean['market'].astype(str).str.strip()
    df_clean['commodity'] = df_clean['commodity'].astype(str).str.strip()
    df_clean['district'] = df_clean['district'].astype(str).str.strip()
    
    # Sort by entity and date, drop exact duplicates
    df_clean.sort_values(by=['commodity', 'market', 'observation_date'], inplace=True)
    df_clean.drop_duplicates(subset=['commodity', 'market', 'observation_date'], keep='last', inplace=True)
    
    return df_clean.reset_index(drop=True)

df_cleaned = clean_and_normalize_data(df_raw)
print(f"✅ Cleaned dataset contains {len(df_cleaned)} valid records.")
print(f"Date Range: {df_cleaned['observation_date'].min().strftime('%Y-%m-%d')} to {df_cleaned['observation_date'].max().strftime('%Y-%m-%d')}")
df_cleaned.head()

### Step 4: Time-Series Feature Engineering Engine

In [ ]:
def build_time_series_features(df):
    df_features = []
    
    # Process each series (commodity + market) independently
    for (comm, mkt), group in df.groupby(['commodity', 'market']):
        group = group.set_index('observation_date').sort_index()
        
        # Reindex to continuous daily frequency to avoid missing date gap issues
        min_d = group.index.min()
        max_d = group.index.max()
        daily_index = pd.date_range(min_d, max_d, freq='D')
        group = group.reindex(daily_index)
        
        # Forward fill price data up to 3 days max
        group['modal_price_orig'] = group['modal_price']
        group['modal_price'] = group['modal_price'].ffill(limit=3)
        group['commodity'] = comm
        group['market'] = mkt
        
        # 1. Price Lags
        for lag in [1, 2, 3, 4, 7, 14, 30]:
            group[f'price_lag_{lag}'] = group['modal_price'].shift(lag)
            
        # 2. Rolling Window Averages & Volatility
        for w in [3, 7, 14, 30]:
            group[f'price_roll_mean_{w}'] = group['modal_price'].shift(1).rolling(w, min_periods=1).mean()
            group[f'price_roll_std_{w}'] = group['modal_price'].shift(1).rolling(w, min_periods=1).std()
            group[f'price_roll_min_{w}'] = group['modal_price'].shift(1).rolling(w, min_periods=1).min()
            group[f'price_roll_max_{w}'] = group['modal_price'].shift(1).rolling(w, min_periods=1).max()
            
        # 3. Short vs Long Trend Ratio
        group['trend_7_30_ratio'] = group['price_roll_mean_7'] / (group['price_roll_mean_30'] + 1e-5)
        
        # 4. Multi-Horizon Direct Targets
        group['target_h1'] = group['modal_price_orig'].shift(-1)  # Day +1
        group['target_h2'] = group['modal_price_orig'].shift(-2)  # Day +2
        group['target_h3'] = group['modal_price_orig'].shift(-3)  # Day +3
        group['modal_price'] = group['modal_price_orig']
        
        group = group.reset_index().rename(columns={'index': 'observation_date'})
        df_features.append(group)
        
    df_full = pd.concat(df_features, ignore_index=True)
    
    # Calendar & Cyclical Features
    df_full['day_of_week'] = df_full['observation_date'].dt.dayofweek
    df_full['month'] = df_full['observation_date'].dt.month
    df_full['day_of_year'] = df_full['observation_date'].dt.dayofyear
    
    df_full['sin_month'] = np.sin(2 * np.pi * df_full['month'] / 12.0)
    df_full['cos_month'] = np.cos(2 * np.pi * df_full['month'] / 12.0)
    df_full['sin_dayofweek'] = np.sin(2 * np.pi * df_full['day_of_week'] / 7.0)
    df_full['cos_dayofweek'] = np.cos(2 * np.pi * df_full['day_of_week'] / 7.0)
    
    # Regional / Cross-Market Average Feature
    mkt_avg = df_full.groupby(['commodity', 'observation_date'])['modal_price'].transform('mean')
    df_full['regional_commodity_mean_price'] = mkt_avg
    
    return df_full

df_ml = build_time_series_features(df_cleaned)
print(f"✅ Feature Engineering Complete. Generated {df_ml.shape[1]} columns.")
df_ml.dropna(subset=['price_lag_1', 'modal_price'], inplace=True)
print(f"Usable feature dataset size: {len(df_ml)} rows.")

### Step 5: Chronological Train / Test Split

In [ ]:
# Chronological cutoff: Train (before 2026), Test (2026+)
train_df = df_ml[df_ml['observation_date'] < '2026-01-01'].copy()
test_df = df_ml[(df_ml['observation_date'] >= '2026-01-01') & (df_ml['target_h1'].notnull())].copy()

feature_cols = [
    'modal_price', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_4', 'price_lag_7', 'price_lag_14', 'price_lag_30',
    'price_roll_mean_3', 'price_roll_std_3', 'price_roll_min_3', 'price_roll_max_3',
    'price_roll_mean_7', 'price_roll_std_7', 'price_roll_min_7', 'price_roll_max_7',
    'price_roll_mean_14', 'price_roll_std_14', 'price_roll_mean_30', 'price_roll_std_30',
    'trend_7_30_ratio', 'day_of_week', 'month', 'sin_month', 'cos_month', 'sin_dayofweek', 'cos_dayofweek',
    'regional_commodity_mean_price'
]

cat_cols = ['market', 'commodity']

print(f"Training set: {len(train_df)} records ({train_df['observation_date'].min().strftime('%Y-%m-%d')} to {train_df['observation_date'].max().strftime('%Y-%m-%d')})")
print(f"Testing set:  {len(test_df)} records ({test_df['observation_date'].min().strftime('%Y-%m-%d')} to {test_df['observation_date'].max().strftime('%Y-%m-%d')})")

### Step 6: Train Multi-Horizon CatBoost Models & Conformal Residuals

In [ ]:
horizons = {'h1': 'target_h1', 'h2': 'target_h2', 'h3': 'target_h3'}
models = {}
conformal_residuals = {}
test_predictions = {}

# Split training set into fit set (80%) and held-out calibration set (20%)
calib_size = int(len(train_df) * 0.20)
fit_df = train_df.iloc[:-calib_size].copy()
calib_df = train_df.iloc[-calib_size:].copy()

X_fit = fit_df[feature_cols + cat_cols].fillna(0)
X_calib = calib_df[feature_cols + cat_cols].fillna(0)
X_test = test_df[feature_cols + cat_cols].fillna(0)

for h_name, target_col in horizons.items():
    print(f"\n🚀 Training CatBoost Model for {h_name.upper()} ({target_col})...")
    
    valid_fit_idx = fit_df[target_col].notnull()
    X_f = X_fit[valid_fit_idx]
    y_f = fit_df.loc[valid_fit_idx, target_col]

    valid_calib_idx = calib_df[target_col].notnull()
    X_c = X_calib[valid_calib_idx]
    y_c = calib_df.loc[valid_calib_idx, target_col]
    
    fit_pool = Pool(X_f, y_f, cat_features=cat_cols)
    calib_pool = Pool(X_c, y_c, cat_features=cat_cols)
    
    model = CatBoostRegressor(
        iterations=600,
        learning_rate=0.04,
        depth=6,
        loss_function='RMSE',
        random_seed=42,
        verbose=100
    )
    
    model.fit(fit_pool, eval_set=calib_pool, early_stopping_rounds=50)
    models[h_name] = model
    
    # Calculate empirical residuals on held-out calibration set for 80% conformal interval
    calib_preds = model.predict(calib_pool)
    residuals = np.abs(y_c.values - calib_preds)
    q80 = np.quantile(residuals, 0.80)
    conformal_residuals[h_name] = float(q80)
    
    # Predict on test set
    test_pool = Pool(X_test, cat_features=cat_cols)
    test_predictions[h_name] = model.predict(test_pool)
    print(f"✅ {h_name.upper()} Training Complete! 80% Conformal Margin: ±₹{q80:.2f}/qtl")

### Step 7: Model Evaluation Metrics & Benchmarks

### Step 8: Visualization Suite

In [ ]:
# 1. Plot Actual vs Forecasted Prices for Test Set
plt.figure(figsize=(14, 6))
sample_test = test_df[(test_df['commodity'] == 'Tomato') & (test_df['market'] == 'Madanapalli APMC')].sort_values('observation_date').head(60).copy()
sample_y = sample_test['target_h1'].values
sample_p = test_predictions['h1'][sample_test.index] if len(test_predictions['h1']) == len(test_df) else test_predictions['h1'][:len(sample_test)]
margin_h1 = conformal_residuals['h1']

plt.plot(sample_test['observation_date'], sample_y, label='Actual Modal Price (₹/qtl)', color='#10b981', linewidth=2, marker='o')
plt.plot(sample_test['observation_date'], sample_p, label='CatBoost H1 Forecast (₹/qtl)', color='#3b82f6', linewidth=2, linestyle='--', marker='s')
plt.fill_between(sample_test['observation_date'], sample_p - margin_h1, sample_p + margin_h1, color='#3b82f6', alpha=0.15, label='80% Conformal Interval')

plt.title('CropMandi AI – 1-Day Horizon Price Forecast vs Actuals', fontsize=14, fontweight='bold')
plt.xlabel('Observation Date')
plt.ylabel('Price (₹ per quintal)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# 2. Feature Importance Plot for H1 Model
feature_importances = models['h1'].get_feature_importance()
fi_df = pd.DataFrame({'feature': feature_cols + cat_cols, 'importance': feature_importances}).sort_values('importance', ascending=False).head(12)

plt.figure(figsize=(10, 5))
sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis')
plt.title('Top 12 Features Driving Price Forecast (Horizon 1)', fontsize=14, fontweight='bold')
plt.xlabel('CatBoost Feature Importance (%)')
plt.tight_layout()
plt.show()

### Step 9: Inference & Farmer Decision Support Advisory Engine

In [ ]:
def generate_farmer_3day_forecast(commodity, market, prediction_date_str):
    pred_dt = pd.to_datetime(prediction_date_str)
    
    # Filter historical data for this market & crop
    sub = df_ml[(df_ml['commodity'] == commodity) & (df_ml['market'] == market)].sort_values('observation_date')
    if len(sub) == 0:
        return f"❌ No data available for '{commodity}' at '{market}'."
        
    latest_row = sub.iloc[-1]
    latest_price = latest_row['modal_price']
    latest_date = latest_row['observation_date'].strftime('%Y-%m-%d')
    
    # Prepare input feature row
    input_row = pd.DataFrame([latest_row[feature_cols + cat_cols]]).fillna(0)
    pool = Pool(input_row, cat_features=cat_cols)
    
    predictions = []
    for i, h_name in enumerate(['h1', 'h2', 'h3'], 1):
        pred_val = float(models[h_name].predict(pool)[0])
        margin = conformal_residuals[h_name]
        base_dt = latest_row['observation_date']
        target_date = (base_dt + timedelta(days=i)).strftime('%Y-%m-%d')
        predictions.append({
            'day': f'Day {i}',
            'target_date': target_date,
            'predicted_price_qtl': round(pred_val, 2),
            'lower_bound_80': round(max(0, pred_val - margin), 2),
            'upper_bound_80': round(pred_val + margin, 2)
        })
        
    # Compute expected trend
    h1_p = predictions[0]['predicted_price_qtl']
    h3_p = predictions[2]['predicted_price_qtl']
    pct_change = round(((h3_p - latest_price) / latest_price) * 100, 2)
    trend = 'UPWARD 📈' if pct_change > 2 else ('DOWNWARD 📉' if pct_change < -2 else 'STABLE ➡️')
    
    # Farmer Advisory Logic
    max_p = max(p['predicted_price_qtl'] for p in predictions)
    max_day = [p['predicted_price_qtl'] for p in predictions].index(max_p) + 1
    gain = round(max_p - latest_price)
    
    if gain > 50 and max_day > 1:
        advisory = f"💡 Hold harvest until Day {max_day} ({predictions[max_day-1]['target_date']}) to gain +₹{gain}/qtl extra profit."
    elif pct_change < -2:
        advisory = f"💡 Prices expected to drop by {abs(pct_change)}%. Sell early (Day 1) to maximize returns before further market arrivals."
    else:
        advisory = f"💡 Prices expected to remain stable around ₹{round(h1_p)}/qtl. Sell based on transport convenience."
        
    result = {
        'commodity': commodity,
        'market': market,
        'latest_observed_price': f'₹{latest_price}/qtl ({latest_date})',
        '3d_trend': f'{trend} ({pct_change}% change)',
        'advisory': advisory,
        'forecasts': predictions,
        'disclaimer': 'Mandatory Decision Support Warning: Prices depend on demand, arrivals, weather, and transport conditions. Use forecast as decision support, not guaranteed selling price.'
    }
    return result

# Run Live Inference Test
demo_result = generate_farmer_3day_forecast('Tomato', 'Madanapalli APMC', '2026-08-13')
print(json.dumps(demo_result, indent=2))

### Step 10: Export Model Artifacts

In [ ]:
os.makedirs("catboost_models", exist_ok=True)

for h_name, model in models.items():
    model_path = f"catboost_models/catboost_{h_name}.cbm"
    model.save_model(model_path)
    print(f"Saved {h_name} model to {model_path}")
    
meta = {
    'model_version': 'v' + datetime.now().strftime('%Y%m%d_%H%M%S'),
    'conformal_residuals': conformal_residuals,
    'feature_cols': feature_cols,
    'cat_cols': cat_cols,
    'metrics': eval_results
}

with open("catboost_models/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
    
print("\n✅ All CatBoost model artifacts & metadata saved successfully!")